In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.store")

#Data Transform

In [0]:
df.display()

##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

##standardize date 

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("store_open_date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("store_close_date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

In [0]:
date_cols = ["store_open_date", "store_close_date"]
formats = ["yyyy-MM-dd", "yyyy-dd-MM"]

def parse_date(col_name, formats):
    return F.coalesce(*[F.try_to_date(F.col(col_name), f) for f in formats])

for c in date_cols:
    df = df.withColumn(c, parse_date(c, formats))

In [0]:
df.display()

In [0]:
df = df.withColumn(
    "store_availability",
    F.when(F.col("store_close_date").isNull(), "available")
     .otherwise("not available")
)

In [0]:
df.display()

In [0]:
df.display()

##De-duplicate

In [0]:
dups = (df.groupBy(df.columns)
          .count()
          .filter("count > 1")
          .orderBy(F.desc("count")))

dups.show(1000, truncate=False)
print("duplicated groups:", dups.count())

In [0]:
w = Window.partitionBy(*df.columns).orderBy(F.lit(1))
df = df.withColumn("row_count", F.row_number().over(w))
df = df.filter("row_count = 1").drop("row_count") 

#Write Into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.store")
)

In [0]:
%sql
select * from silver.store